# QTDB 3600 — một Conditional DDPM cho nhiễu trộn BW+EM+MA

Huấn luyện **mới từ epoch 0** bằng kiến trúc HNF–FiLM–attention và ResearchLossDDPM của nghiên cứu. Mỗi cặp train/validation được tạo bằng công thức trộn đồng thời ba nguồn NSTDB BW, EM, MA (n_type=1); không train ba model riêng. Tín hiệu sạch và test cố định lấy từ bộ common QTDB 3600. Không nạp checkpoint của lần chạy cũ trong phiên đầu.

Các hệ số biên `a1` hoặc `a2` bằng 0/100 có thể làm một thành phần bằng 0 ở một số ví dụ; đây là đúng miền hệ số của giao thức TFCDiff, không phải lỗi chọn một loại nhiễu thay vì trộn.


## 1. Input Kaggle

- Gắn Dataset chứa `common_qtdb_mixed_bwem_ma_3600_ntype1.npz`: notebook dùng `clean_train`, `clean_val`, `clean_test`, `noisy_test`, ID và mức `q`. Không dùng `noisy_train/noisy_val` trong NPZ để train; chúng được sinh lại từ ba nguồn nhiễu trong cell dữ liệu.
- Bật GPU và Internet để đọc NSTDB qua WFDB. Lần chạy đầu **không cần `last_*.pth`**; để `START_FROM_SCRATCH=True`.
- Bộ test giữ nguyên `noisy_test`/`clean_test`/`sample_id_test` trong NPZ. Nếu dùng kết quả với các model khác, chúng phải đánh giá trên đúng cặp test này và cùng script metric.

Việc tạo lại nhiễu train làm cặp train khác với bản NPZ đóng băng; do đó đây là so sánh trên **cùng test**, không khẳng định cùng chính xác cặp train với checkpoint TFCDiff lịch sử.


## 2. Cấu hình nghiên cứu và giao thức

ECG 1 kênh, 360 Hz, 3600 mẫu. Với `n_type=1`, mỗi nguồn BW/EM/MA lấy kênh 1, nửa đầu cho train/validation; test cố định trong NPZ. Mỗi clean segment được ghép một cặp nhiễu duy nhất trước khi huấn luyện, dùng seed cố định. Công thức trộn giữ nguyên dấu ngoặc và chỉ chia 100 một lần theo `compare.md`. Mức `q` lấy từ manifest NPZ. Mô hình: U-Net HNF–FiLM–attention, DDPM 50 bước, ResearchLossDDPM, Adam 1e-4, EMA 0.9.


In [ ]:
%pip -q install 'wfdb>=4.1,<5'

In [ ]:
from pathlib import Path
import gc
import base64
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
try:
    from IPython.display import display
except ImportError:
    display = print

# ---- Chỉ sửa khối này giữa các phiên Kaggle ----
START_FROM_SCRATCH = True  # True cho run đầu: bỏ qua mọi last checkpoint
MODE = 'train'                 # 'train' hoặc 'export'
REQUIRE_RESUME_CHECKPOINT = False  # phiên tiếp tục từ checkpoint epoch 40; False nếu cố ý train mới
TARGET_EPOCHS = 400
MAX_EPOCHS_THIS_SESSION = 5    # None = cố chạy đến TARGET_EPOCHS
MICRO_BATCH_SIZE = 4           # giảm 2/1 nếu CUDA OOM
REFERENCE_BATCH_SIZE = 96
GRAD_ACCUM_STEPS = REFERENCE_BATCH_SIZE // MICRO_BATCH_SIZE
NUM_WORKERS = 2
USE_AMP = True
EMA_DECAY = 0.9
SEED = 1234                    # model/training RNG
SPLIT_SEED = 1234             # khóa phép chia 70/30 source-train
ALLOW_UNVERIFIED_BUNDLE = True  # tương thích NPZ tối giản; xem cảnh báo data audit

TRAIN_BUNDLE_NAME = 'common_qtdb_mixed_bwem_ma_3600_ntype1.npz'
EXPECTED_MIXED_BUNDLE_SHA256 = '517bfe0b825fadb05f272e73c205508de034f8d3d403e384b1e1a4afb6c26a4f'
TEST_BUNDLE_NAME = TRAIN_BUNDLE_NAME
MANIFEST_NAME = 'mixed_manifest.csv'
RUN_EXPORT_WHEN_COMPLETE = True
ALLOW_INCOMPLETE_EXPORT = False
INFERENCE_BATCH_SIZE = 4
DDIM_STEPS = 50
DDIM_ETA = 0.0
NUM_SHOTS = 1

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/qtdb_mixed_bwem_ma_3600_fresh')
SOURCE_SNAPSHOT = WORK_ROOT / 'source_snapshot'
CHECKPOINT_DIR = WORK_ROOT / 'checkpoints'
REPORT_DIR = WORK_ROOT / 'reports'
for directory in (SOURCE_SNAPSHOT, CHECKPOINT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Bật GPU Accelerator trong Kaggle trước khi chạy notebook.')
if MODE not in {'train', 'export'}:
    raise ValueError(MODE)
if REFERENCE_BATCH_SIZE % MICRO_BATCH_SIZE:
    raise ValueError('REFERENCE_BATCH_SIZE phải chia hết cho MICRO_BATCH_SIZE')
if TARGET_EPOCHS != 400:
    print('WARNING: compare.md mô tả 400 epoch; hiện tại:', TARGET_EPOCHS)
if NUM_SHOTS != 1:
    raise ValueError('Benchmark chính là 1-shot.')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

MODEL_CONFIG = {
    'train': {
        'feats': 80, 'batch_size': REFERENCE_BATCH_SIZE,
        'micro_batch_size': MICRO_BATCH_SIZE, 'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
        'epochs': TARGET_EPOCHS, 'lr': 1e-4, 'weight_decay': 0.0,
        'scheduler': {'name': 'StepLR', 'step_size': 150, 'gamma': 0.1},
        'gradient_clip_norm': 1.0, 'lambda_time': 1.0, 'lambda_freq': 0.1,
    },
    'diffusion': {'beta_start': 1e-4, 'beta_end': 0.5,
                  'num_steps': 50, 'schedule': 'quad'},
    'experiment': {
        'name': 'qtdb_mixed_bwem_ma_3600_fresh_generated_train',
        'input_length': 3600, 'sampling_rate': 360,
        'in_channels': 2, 'out_channels': 1,
        'normalization': 'per_sample_noisy_maxabs_shared_v1',
        'stft_n_fft': 128, 'stft_hop_length': 64,
        'ema_decay': EMA_DECAY, 'seed': SEED, 'split_seed': SPLIT_SEED,
        'data_protocol': 'tfcdiff_mixed_bwem_ma_generated_train', 'n_type': 1,
        'mixed_train_seed': 1234,
        'loss_version': 'l1_mean_relative_stft_snr_weighted_continuous_alpha_v1',
        'research_loss_sha256': '4b444b6e37bfb5385fca4f49e783f447fcd51c17b67d9345b494f4223c773714',
    },
}

print('Device:', torch.cuda.get_device_name(0))
print('Mode:', MODE, '| target epochs:', TARGET_EPOCHS,
      '| max this session:', MAX_EPOCHS_THIS_SESSION)
print('Micro/effective batch:', MICRO_BATCH_SIZE, '/', REFERENCE_BATCH_SIZE,
      '| accumulation:', GRAD_ACCUM_STEPS, '| AMP:', USE_AMP)


## 3. Tái tạo source snapshot nhúng sẵn và nhập model gốc

In [ ]:
REQUIRED_SOURCE_FILES = (
    'src/__init__.py',
    'src/models/__init__.py',
    'src/models/main_model.py',
    'src/models/unet_1d.py',
    'src/models/blocks/__init__.py',
    'src/models/blocks/hnf_block.py',
    'src/models/blocks/bridge_block.py',
    'src/models/blocks/attention_1d.py',
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

working_src = SOURCE_SNAPSHOT / 'src'
if working_src.exists():
    shutil.rmtree(working_src)
EMBEDDED_SOURCE_B64 = json.loads(r"""{"src/__init__.py":"","src/models/__init__.py":"","src/models/blocks/__init__.py":"ZnJvbSAuaG5mX2Jsb2NrIGltcG9ydCBITkZCbG9jaw0KZnJvbSAuYnJpZGdlX2Jsb2NrIGltcG9ydCBCcmlkZ2VCbG9jaw0KZnJvbSAuYXR0ZW50aW9uXzFkIGltcG9ydCBTZWxmQXR0ZW50aW9uMUQ=","src/models/blocks/attention_1d.py":"aW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCg0KY2xhc3MgU2VsZkF0dGVudGlvbjFEKG5uLk1vZHVsZSk6DQogICAgIiIiMUQgU2VsZuKAkUF0dGVudGlvbiAobXVsdGnigJFoZWFkKSBkw7luZyB04bqhaSBib3R0bGVuZWNrLiIiIg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjaGFubmVscywgbnVtX2hlYWRzPTQpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5udW1faGVhZHMgPSBudW1faGVhZHMNCiAgICAgICAgc2VsZi5oZWFkX2RpbSA9IGNoYW5uZWxzIC8vIG51bV9oZWFkcw0KICAgICAgICBhc3NlcnQgc2VsZi5oZWFkX2RpbSAqIG51bV9oZWFkcyA9PSBjaGFubmVscywgImNoYW5uZWxzIHBo4bqjaSBjaGlhIGjhur90IGNobyBudW1faGVhZHMiDQoNCiAgICAgICAgc2VsZi5xa3YgPSBubi5Db252MWQoY2hhbm5lbHMsIGNoYW5uZWxzICogMywga2VybmVsX3NpemU9MSkNCiAgICAgICAgc2VsZi5wcm9qID0gbm4uQ29udjFkKGNoYW5uZWxzLCBjaGFubmVscywga2VybmVsX3NpemU9MSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOg0KICAgICAgICBCLCBDLCBMID0geC5zaGFwZQ0KICAgICAgICBxa3YgPSBzZWxmLnFrdih4KS5yZXNoYXBlKEIsIDMsIHNlbGYubnVtX2hlYWRzLCBzZWxmLmhlYWRfZGltLCBMKQ0KICAgICAgICBxLCBrLCB2ID0gcWt2WzosIDBdLCBxa3ZbOiwgMV0sIHFrdls6LCAyXQ0KICAgICAgICAjIHEsIGssIHY6IChCLCBudW1faGVhZHMsIGhlYWRfZGltLCBMKQ0KICAgICAgICBhdHRuID0gdG9yY2gubWF0bXVsKHEudHJhbnNwb3NlKC0yLCAtMSksIGspIC8gKHNlbGYuaGVhZF9kaW0gKiogMC41KQ0KICAgICAgICBhdHRuID0gdG9yY2guc29mdG1heChhdHRuLCBkaW09LTEpDQogICAgICAgIG91dCA9IHRvcmNoLm1hdG11bChhdHRuLCB2LnRyYW5zcG9zZSgtMiwgLTEpKS50cmFuc3Bvc2UoLTIsIC0xKQ0KICAgICAgICBvdXQgPSBvdXQucmVzaGFwZShCLCBDLCBMKQ0KICAgICAgICByZXR1cm4gc2VsZi5wcm9qKG91dCk=","src/models/blocks/bridge_block.py":"aW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmltcG9ydCBtYXRoDQoNCmNsYXNzIEJyaWRnZUJsb2NrKG5uLk1vZHVsZSk6DQogICAgIiIiRmlMTeKAkWJhc2VkIGNvbmRpdGlvbmluZyBvbiBub2lzZSBsZXZlbCAoc3FydChhbHBoYV9iYXIpKS4iIiINCiAgICBkZWYgX19pbml0X18oc2VsZiwgZmVhdHVyZXMsIGVtYl9kaW09MTI4KToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYuZW1iX2RpbSA9IGVtYl9kaW0NCiAgICAgICAgc2VsZi5maWxtID0gbm4uU2VxdWVudGlhbCgNCiAgICAgICAgICAgIG5uLkxpbmVhcihlbWJfZGltLCBmZWF0dXJlcyAqIDIpLA0KICAgICAgICAgICAgbm4uU2lMVSgpDQogICAgICAgICkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIGFscGhhX2Jhcik6DQogICAgICAgICMgYWxwaGFfYmFyOiAoQiwpIG9yIHNjYWxhcg0KICAgICAgICAjIMOJcCBwaOG6s25nIGFscGhhX2JhciB24buBIGThuqFuZyAxRCAoQmF0Y2gsKSDEkeG7gyB0csOhbmggYuG7iyBkxrAgY2hp4buBdSAtLSBD4buwQyBRVUFOIFRS4buMTkcNCiAgICAgICAgYWxwaGFfYmFyID0gYWxwaGFfYmFyLnZpZXcoLTEpDQogICAgICAgIGVtYiA9IHNlbGYuc2ludXNvaWRhbF9lbWJlZGRpbmcoYWxwaGFfYmFyKQ0KICAgICAgICBzY2FsZV9zaGlmdCA9IHNlbGYuZmlsbShlbWIpICAgIyAoQiwgMipmZWF0dXJlcykNCiAgICAgICAgc2NhbGUsIHNoaWZ0ID0gc2NhbGVfc2hpZnQuY2h1bmsoMiwgZGltPTEpDQogICAgICAgIHNjYWxlID0gc2NhbGUudW5zcXVlZXplKC0xKQ0KICAgICAgICBzaGlmdCA9IHNoaWZ0LnVuc3F1ZWV6ZSgtMSkNCiAgICAgICAgcmV0dXJuIHggKiAoMSArIHNjYWxlKSArIHNoaWZ0DQoNCiAgICBkZWYgc2ludXNvaWRhbF9lbWJlZGRpbmcoc2VsZiwgeCk6DQogICAgICAgIGlmIHguZGltKCkgPT0gMDoNCiAgICAgICAgICAgIHggPSB4LnVuc3F1ZWV6ZSgwKQ0KICAgICAgICBkZXZpY2UgPSB4LmRldmljZQ0KICAgICAgICBoYWxmX2RpbSA9IHNlbGYuZW1iX2RpbSAvLyAyDQogICAgICAgIGVtYiA9IG1hdGgubG9nKDEwMDAwKSAvIChoYWxmX2RpbSAtIDEpDQogICAgICAgIGVtYiA9IHRvcmNoLmV4cCh0b3JjaC5hcmFuZ2UoaGFsZl9kaW0sIGRldmljZT1kZXZpY2UpICogLWVtYikNCiAgICAgICAgZW1iID0geC51bnNxdWVlemUoLTEpICogZW1iLnVuc3F1ZWV6ZSgwKQ0KICAgICAgICBlbWIgPSB0b3JjaC5jYXQoW3RvcmNoLnNpbihlbWIpLCB0b3JjaC5jb3MoZW1iKV0sIGRpbT0tMSkNCiAgICAgICAgcmV0dXJuIGVtYg==","src/models/blocks/hnf_block.py":"aW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCg0KY2xhc3MgSE5GQmxvY2sobm4uTW9kdWxlKToNCiAgICAiIiINCiAgICBIYWxmIE5vcm1hbGl6ZWQgRmlsdGVyIGJsb2NrLg0KICAgIE11bHRpLXNjYWxlIGNvbnZzIOKGkiBjb25jYXQg4oaSIGNvbnYg4oaSIGhhbGbigJFpbnN0YW5jZSBub3JtIOKGkiByZXNpZHVhbC4NCiAgICAiIiINCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2hhbm5lbHMsIG91dF9jaGFubmVscywga2VybmVsX3NpemVzPSgzLCA1LCA5LCAxNSkpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5tdWx0aV9jb252cyA9IG5uLk1vZHVsZUxpc3QoWw0KICAgICAgICAgICAgbm4uQ29udjFkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMgLy8gbGVuKGtlcm5lbF9zaXplcyksIGssDQogICAgICAgICAgICAgICAgICAgICAgcGFkZGluZz1rLy8yLCBwYWRkaW5nX21vZGU9J3JlZmxlY3QnKQ0KICAgICAgICAgICAgZm9yIGsgaW4ga2VybmVsX3NpemVzDQogICAgICAgIF0pDQogICAgICAgIHNlbGYuYWdnX2NvbnYgPSBubi5Db252MWQob3V0X2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDEpDQogICAgICAgICMgSGFsZuKAkWluc3RhbmNlIG5vcm06IGNoaWEgY2hhbm5lbCBsw6BtIDIgbuG7rWENCiAgICAgICAgc2VsZi5oYWxmX2luc3Rfbm9ybSA9IG5uLkluc3RhbmNlTm9ybTFkKG91dF9jaGFubmVscyAvLyAyKQ0KICAgICAgICBzZWxmLmFjdCA9IG5uLlJlTFUoaW5wbGFjZT1UcnVlKQ0KICAgICAgICBzZWxmLnJlc2lkdWFsID0gbm4uQ29udjFkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDEpIGlmIGluX2NoYW5uZWxzICE9IG91dF9jaGFubmVscyBlbHNlIG5uLklkZW50aXR5KCkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOg0KICAgICAgICBtdWx0aV9vdXQgPSBbY29udih4KSBmb3IgY29udiBpbiBzZWxmLm11bHRpX2NvbnZzXQ0KICAgICAgICBvdXQgPSB0b3JjaC5jYXQobXVsdGlfb3V0LCBkaW09MSkNCiAgICAgICAgb3V0ID0gc2VsZi5hZ2dfY29udihvdXQpDQogICAgICAgIGhhbGYgPSBvdXQuc2hhcGVbMV0gLy8gMg0KICAgICAgICBvdXRfbm9ybSA9IHNlbGYuaGFsZl9pbnN0X25vcm0ob3V0WzosIDpoYWxmLCA6XSkNCiAgICAgICAgb3V0ID0gdG9yY2guY2F0KFtvdXRfbm9ybSwgb3V0WzosIGhhbGY6LCA6XV0sIGRpbT0xKQ0KICAgICAgICBvdXQgPSBzZWxmLmFjdChvdXQpDQogICAgICAgIHJldHVybiBvdXQgKyBzZWxmLnJlc2lkdWFsKHgp","src/models/main_model.py":"...","src/models/unet_1d.py":"aW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmZyb20gLmJsb2NrcyBpbXBvcnQgSE5GQmxvY2ssIEJyaWRnZUJsb2NrLCBTZWxmQXR0ZW50aW9uMUQNCg0KY2xhc3MgVU5ldDFEKG5uLk1vZHVsZSk6DQogICAgIiIiDQogICAgMUQgVeKAkU5ldCB24bubaSA0IGxldmVscywgc2tpcCBjb25uZWN0aW9ucywgSE5GIGJsb2NrcywNCiAgICBCcmlkZ2UgRmlMTSAodGltZXN0ZXApLCB2w6AgU2VsZuKAkUF0dGVudGlvbiB04bqhaSBib3R0bGVuZWNrLg0KICAgICIiIg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFubmVscz0yNCwgYmFzZV9jaGFubmVscz02NCwgZW1iX2RpbT0xMjgsIG91dF9jaGFubmVscz0xMik6DQogICAgICAgICIiIg0KICAgICAgICBpbl9jaGFubmVscyA9IDI0IHbDrCBjb25jYXQgeF90ICgxMiBjaGFubmVscykgdsOgIGNvbmRpdGlvbiAoMTIgY2hhbm5lbHMpDQogICAgICAgICIiIg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgIyAtLS0tLS0tLS0tIEVOQ09ERVIgLS0tLS0tLS0tLQ0KICAgICAgICAjIExldmVsIDEgKGlucHV0IC0+IDY0KQ0KICAgICAgICBzZWxmLmVuYzEgPSBITkZCbG9jayhpbl9jaGFubmVscywgYmFzZV9jaGFubmVscykNCiAgICAgICAgc2VsZi5icmlkZ2UxID0gQnJpZGdlQmxvY2soYmFzZV9jaGFubmVscywgZW1iX2RpbSkNCiAgICAgICAgc2VsZi5kb3duMSA9IG5uLkNvbnYxZChiYXNlX2NoYW5uZWxzLCBiYXNlX2NoYW5uZWxzKjIsIGtlcm5lbF9zaXplPTQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQoNCiAgICAgICAgIyBMZXZlbCAyICg2NCAtPiAxMjgpDQogICAgICAgIHNlbGYuZW5jMiA9IEhORkJsb2NrKGJhc2VfY2hhbm5lbHMqMiwgYmFzZV9jaGFubmVscyoyKQ0KICAgICAgICBzZWxmLmJyaWRnZTIgPSBCcmlkZ2VCbG9jayhiYXNlX2NoYW5uZWxzKjIsIGVtYl9kaW0pDQogICAgICAgIHNlbGYuZG93bjIgPSBubi5Db252MWQoYmFzZV9jaGFubmVscyoyLCBiYXNlX2NoYW5uZWxzKjQsIGtlcm5lbF9zaXplPTQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQoNCiAgICAgICAgIyBMZXZlbCAzICgxMjggLT4gMjU2KQ0KICAgICAgICBzZWxmLmVuYzMgPSBITkZCbG9jayhiYXNlX2NoYW5uZWxzKjQsIGJhc2VfY2hhbm5lbHMqNCkNCiAgICAgICAgc2VsZi5icmlkZ2UzID0gQnJpZGdlQmxvY2soYmFzZV9jaGFubmVscyo0LCBlbWJfZGltKQ0KICAgICAgICBzZWxmLmRvd24zID0gbm4uQ29udjFkKGJhc2VfY2hhbm5lbHMqNCwgYmFzZV9jaGFubmVscyo4LCBrZXJuZWxfc2l6ZT00LCBzdHJpZGU9MiwgcGFkZGluZz0xKQ0KDQogICAgICAgICMgTGV2ZWwgNCAoMjU2IC0+IDUxMikg4oCTIGJvdHRsZW5lY2sNCiAgICAgICAgc2VsZi5lbmM0ID0gSE5GQmxvY2soYmFzZV9jaGFubmVscyo4LCBiYXNlX2NoYW5uZWxzKjgpDQogICAgICAgIHNlbGYuYnJpZGdlNCA9IEJyaWRnZUJsb2NrKGJhc2VfY2hhbm5lbHMqOCwgZW1iX2RpbSkNCiAgICAgICAgIyBTZWxm4oCRQXR0ZW50aW9uIHThuqFpIGJvdHRsZW5lY2sNCiAgICAgICAgc2VsZi5hdHRuID0gU2VsZkF0dGVudGlvbjFEKGJhc2VfY2hhbm5lbHMqOCkNCg0KICAgICAgICAjIC0tLS0tLS0tLS0gREVDT0RFUiAtLS0tLS0tLS0tDQogICAgICAgICMgTGV2ZWwgNCAtPiAzDQogICAgICAgIHNlbGYudXA0ID0gbm4uQ29udlRyYW5zcG9zZTFkKGJhc2VfY2hhbm5lbHMqOCwgYmFzZV9jaGFubmVscyo0LCBrZXJuZWxfc2l6ZT00LCBzdHJpZGU9MiwgcGFkZGluZz0xKQ0KICAgICAgICBzZWxmLmRlYzQgPSBITkZCbG9jayhiYXNlX2NoYW5uZWxzKjgsIGJhc2VfY2hhbm5lbHMqNCkgICAjIGNvbmNhdCB24bubaSBza2lwIHThu6sgZW5jMw0KDQogICAgICAgICMgTGV2ZWwgMyAtPiAyDQogICAgICAgIHNlbGYudXAzID0gbm4uQ29udlRyYW5zcG9zZTFkKGJhc2VfY2hhbm5lbHMqNCwgYmFzZV9jaGFubmVscyoyLCBrZXJuZWxfc2l6ZT00LCBzdHJpZGU9MiwgcGFkZGluZz0xKQ0KICAgICAgICBzZWxmLmRlYzMgPSBITkZCbG9jayhiYXNlX2NoYW5uZWxzKjQsIGJhc2VfY2hhbm5lbHMqMikgICAjIGNvbmNhdCB24bubaSBza2lwIHThu6sgZW5jMg0KDQogICAgICAgICMgTGV2ZWwgMiAtPiAxDQogICAgICAgIHNlbGYudXAyID0gbm4uQ29udlRyYW5zcG9zZTFkKGJhc2VfY2hhbm5lbHMqMiwgYmFzZV9jaGFubmVscywga2VybmVsX3NpemU9NCwgc3RyaWRlPTIsIHBhZGRpbmc9MSkNCiAgICAgICAgc2VsZi5kZWMyID0gSE5GQmxvY2soYmFzZV9jaGFubmVscyoyLCBiYXNlX2NoYW5uZWxzKSAgICAgIyBjb25jYXQgduG7m2kgc2tpcCB04burIGVuYzENCg0KICAgICAgICAjIE91dHB1dA0KICAgICAgICBzZWxmLmZpbmFsID0gbm4uQ29udjFkKGJhc2VfY2hhbm5lbHMsIG91dF9jaGFubmVscywga2VybmVsX3NpemU9MSkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIGNvbmQsIG5vaXNlX3NjYWxlKToNCiAgICAgICAgIiIiDQogICAgICAgIHg6IGxhdGVudCAoeF90KSwgY29uZDogbm9pc3kgb2JzZXJ2YXRpb24sIG5vaXNlX3NjYWxlOiBzcXJ0KGFscGhhX2JhcikNCiAgICAgICAgIiIiDQogICAgICAgICMgQ29uY2F0ZW5hdGUgeCBhbmQgY29uZCBhbG9uZyBjaGFubmVsIGRpbWVuc2lvbg0KICAgICAgICBpbnAgPSB0b3JjaC5jYXQoW3gsIGNvbmRdLCBkaW09MSkgICAjIChCLCAyLCBMKQ0KDQogICAgICAgICMgLS0tLS0gRW5jb2RlciAtLS0tLQ0KICAgICAgICBlMSA9IHNlbGYuZW5jMShpbnApDQogICAgICAgIGUxID0gc2VsZi5icmlkZ2UxKGUxLCBub2lzZV9zY2FsZSkNCiAgICAgICAgZDEgPSBzZWxmLmRvd24xKGUxKQ0KDQogICAgICAgIGUyID0gc2VsZi5lbmMyKGQxKQ0KICAgICAgICBlMiA9IHNlbGYuYnJpZGdlMihlMiwgbm9pc2Vfc2NhbGUpDQogICAgICAgIGQyID0gc2VsZi5kb3duMihlMikNCg0KICAgICAgICBlMyA9IHNlbGYuZW5jMyhkMikNCiAgICAgICAgZTMgPSBzZWxmLmJyaWRnZTMoZTMsIG5vaXNlX3NjYWxlKQ0KICAgICAgICBkMyA9IHNlbGYuZG93bjMoZTMpDQoNCiAgICAgICAgZTQgPSBzZWxmLmVuYzQoZDMpDQogICAgICAgIGU0ID0gc2VsZi5icmlkZ2U0KGU0LCBub2lzZV9zY2FsZSkNCiAgICAgICAgIyBCb3R0bGVuZWNrIHNlbGbigJFhdHRlbnRpb24NCiAgICAgICAgZTQgPSBzZWxmLmF0dG4oZTQpDQoNCiAgICAgICAgIyAtLS0tLSBEZWNvZGVyICh3aXRoIHNraXAgY29ubmVjdGlvbnMpIC0tLS0tDQogICAgICAgIHU0ID0gc2VsZi51cDQoZTQpDQogICAgICAgIHU0ID0gdG9yY2guY2F0KFt1NCwgZTNdLCBkaW09MSkgICAjIHNraXAgdOG7qyBlbmMzDQogICAgICAgIGQ0ID0gc2VsZi5kZWM0KHU0KQ0KDQogICAgICAgIHUzID0gc2VsZi51cDMoZDQpDQogICAgICAgIHUzID0gdG9yY2guY2F0KFt1MywgZTJdLCBkaW09MSkgICAjIHNraXAgdOG7qyBlbmMyDQogICAgICAgIGQzID0gc2VsZi5kZWMzKHUzKQ0KDQogICAgICAgIHUyID0gc2VsZi51cDIoZDMpDQogICAgICAgIHUyID0gdG9yY2guY2F0KFt1MiwgZTFdLCBkaW09MSkgICAjIHNraXAgdOG7qyBlbmMxDQogICAgICAgIGQyID0gc2VsZi5kZWMyKHUyKQ0KDQogICAgICAgIG91dCA9IHNlbGYuZmluYWwoZDIpDQogICAgICAgIHJldHVybiBvdXQ="}""")
if set(EMBEDDED_SOURCE_B64) != set(REQUIRED_SOURCE_FILES):
    raise RuntimeError('Embedded source manifest mismatch')
for relative, payload in EMBEDDED_SOURCE_B64.items():
    destination = SOURCE_SNAPSHOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(payload))

source_hashes = {name: sha256_file(SOURCE_SNAPSHOT / name) for name in REQUIRED_SOURCE_FILES}
(REPORT_DIR / 'source_hashes.json').write_text(
    json.dumps(source_hashes, indent=2, ensure_ascii=False), encoding='utf-8')
sys.path.insert(0, str(SOURCE_SNAPSHOT))

from src.models.unet_1d import UNet1D
from src.models.main_model import DDPM, EMA, stft_loss

print('Imported byte-identical source embedded in this notebook')
display(pd.DataFrame([{'file': key, 'sha256': value} for key, value in source_hashes.items()]))


## 4. Nạp và khóa common mixed-noise tensors

Cell này kiểm tra protocol, record leakage, shape, q và sample ID trước khi model nhìn thấy dữ liệu. Nếu input là source-train, phép chia 70/30 được tái lập deterministically và chỉ số được xuất ra để mọi mô hình dùng lại.


In [ ]:
TEST_RECORDS = {
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853', 'sel16420',
    'sel16795', 'sele0106', 'sele0121', 'sel32', 'sel49', 'sel14046', 'sel15814',
}

def unique_input(filename, required=True):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if len(matches) > 1:
        raise RuntimeError(f'Nhiều file {filename}: {matches}')
    if required and not matches:
        raise FileNotFoundError(
            f'Không tìm thấy {filename}. Không dùng common_qtdb_bw_3600.npz vì đó là BW-only.')
    return matches[0] if matches else None

def as_ncl3600(array, name):
    value = np.asarray(array)
    if value.ndim == 2 and value.shape[1] == 3600:
        value = value[:, None, :]
    elif value.ndim == 3 and value.shape[1:] == (3600, 1):
        value = value.transpose(0, 2, 1)
    if value.ndim != 3 or value.shape[1:] != (1, 3600):
        raise ValueError(f'{name}: cần [N,1,3600], nhận {value.shape}')
    value = np.asarray(value, dtype=np.float32)
    if not np.isfinite(value).all():
        raise ValueError(f'{name}: có NaN/Inf')
    return value

def scalar(bundle, key, default=None):
    if key not in bundle.files:
        if default is not None:
            return default
        raise KeyError(f'NPZ thiếu metadata bắt buộc: {key}')
    return np.asarray(bundle[key]).reshape(-1)[0].item()

def vector(bundle, key, count, dtype=str, default=None):
    if key not in bundle.files:
        if default is None:
            raise KeyError(f'NPZ thiếu {key}')
        value = np.asarray(default)
    else:
        value = np.asarray(bundle[key]).reshape(-1)
    value = value.astype(dtype)
    if len(value) != count:
        raise ValueError(f'{key}: {len(value)} != {count}')
    return value

train_bundle_path = unique_input(TRAIN_BUNDLE_NAME)
actual_bundle_sha256 = sha256_file(train_bundle_path)
if actual_bundle_sha256 != EXPECTED_MIXED_BUNDLE_SHA256:
    raise RuntimeError(
        f'Wrong mixed bundle SHA-256: {actual_bundle_sha256}; expected '
        f'{EXPECTED_MIXED_BUNDLE_SHA256}. Do not attach BW-only or regenerated pairs.')
train_bundle = np.load(train_bundle_path, allow_pickle=True)
metadata_keys = {'protocol_name', 'n_type', 'sampling_rate', 'segment_length'}
protocol_verified = metadata_keys.issubset(train_bundle.files)
if not protocol_verified and not ALLOW_UNVERIFIED_BUNDLE:
    raise KeyError(f'NPZ thiếu metadata {sorted(metadata_keys-set(train_bundle.files))}')
protocol_name = str(scalar(train_bundle, 'protocol_name', 'tfcdiff_mixed_bwem_ma')).lower()
n_type = int(scalar(train_bundle, 'n_type', 1))
sampling_rate = int(scalar(train_bundle, 'sampling_rate', 360))
segment_length = int(scalar(train_bundle, 'segment_length', 3600))
if protocol_name != 'tfcdiff_mixed_bwem_ma':
    raise ValueError(f'protocol_name={protocol_name!r}; cần tfcdiff_mixed_bwem_ma')
if n_type != 1:
    raise ValueError('Kết quả chính trong compare.md yêu cầu n_type=1')
if sampling_rate != 360 or segment_length != 3600:
    raise ValueError(f'Cần 360 Hz/3600 mẫu; nhận {sampling_rate} Hz/{segment_length}')
if not protocol_verified:
    print('WARNING: NPZ không có protocol metadata; đang chạy compatibility mode.')
    print('WARNING: chỉ tiếp tục nếu file này thực sự là mixed BW+EM+MA n_type=1 theo compare.md.')
    print('Available keys:', train_bundle.files)

has_presplit = {'clean_train', 'noisy_train', 'clean_val', 'noisy_val'}.issubset(train_bundle.files)
has_source = {'clean_source_train', 'noisy_source_train'}.issubset(train_bundle.files)
if has_presplit == has_source:
    raise ValueError('NPZ phải có đúng một dạng: pre-split hoặc source-train chưa chia')

def make_frame(bundle, suffix, count, split):
    generated_ids = np.asarray([f'{split}:{index:08d}' for index in range(count)])
    sample_id = vector(bundle, f'sample_id_{suffix}', count, str, generated_ids)
    inferred_records = np.asarray([
        value.split(':')[1] if len(value.split(':')) >= 3 else 'unknown'
        for value in sample_id
    ])
    record_id = vector(bundle, f'record_id_{suffix}', count, str, inferred_records)
    q = vector(bundle, f'q_{suffix}', count, np.float32,
               np.full(count, np.nan, dtype=np.float32))
    if len(set(sample_id.tolist())) != count:
        raise ValueError(f'sample_id_{suffix} bị trùng')
    finite_q = np.isfinite(q)
    if (((q < .20) | (q > 1.99)) & finite_q).any():
        raise ValueError(f'q_{suffix} ngoài [0.20, 1.99]')
    return pd.DataFrame({'sample_id': sample_id, 'record_id': record_id,
                         'q': q, 'split': split, 'array_row': np.arange(count)})

if has_presplit:
    CLEAN_TRAIN = as_ncl3600(train_bundle['clean_train'], 'clean_train')
    NOISY_TRAIN = as_ncl3600(train_bundle['noisy_train'], 'noisy_train')
    CLEAN_VAL = as_ncl3600(train_bundle['clean_val'], 'clean_val')
    NOISY_VAL = as_ncl3600(train_bundle['noisy_val'], 'noisy_val')
    TRAIN_MANIFEST = make_frame(train_bundle, 'train', len(CLEAN_TRAIN), 'train')
    VAL_MANIFEST = make_frame(train_bundle, 'val', len(CLEAN_VAL), 'validation')
    split_origin = 'pre_split_input'
    split_indices_path = None
    known_bw_only_shape = (len(CLEAN_TRAIN), len(CLEAN_VAL), len(train_bundle['clean_test']))
    if known_bw_only_shape == (5728, 2418, 1252) and not protocol_verified:
        raise RuntimeError(
            'Bundle có đúng fingerprint shape (5728,2418,1252) của common_qtdb_bw_3600.npz BW-only. '
            'Không thể dùng file này cho benchmark mixed BW+EM+MA; cần tạo/upload common mixed-noise thật.')
else:
    clean_source = as_ncl3600(train_bundle['clean_source_train'], 'clean_source_train')
    noisy_source = as_ncl3600(train_bundle['noisy_source_train'], 'noisy_source_train')
    if clean_source.shape != noisy_source.shape:
        raise ValueError('source clean/noisy shape mismatch')
    source_manifest = make_frame(train_bundle, 'source_train', len(clean_source), 'source_train')
    split_rng = np.random.RandomState(SPLIT_SEED)
    permutation = split_rng.permutation(len(clean_source))
    val_count = int(np.ceil(0.30 * len(clean_source)))
    val_indices, train_indices = permutation[:val_count], permutation[val_count:]
    CLEAN_TRAIN, NOISY_TRAIN = clean_source[train_indices], noisy_source[train_indices]
    CLEAN_VAL, NOISY_VAL = clean_source[val_indices], noisy_source[val_indices]
    TRAIN_MANIFEST = source_manifest.iloc[train_indices].copy().reset_index(drop=True)
    VAL_MANIFEST = source_manifest.iloc[val_indices].copy().reset_index(drop=True)
    TRAIN_MANIFEST['split'] = 'train'; TRAIN_MANIFEST['array_row'] = np.arange(len(TRAIN_MANIFEST))
    VAL_MANIFEST['split'] = 'validation'; VAL_MANIFEST['array_row'] = np.arange(len(VAL_MANIFEST))
    split_indices_path = REPORT_DIR / 'split_indices.npz'
    np.savez(split_indices_path, train_indices=train_indices, val_indices=val_indices,
             split_seed=np.asarray(SPLIT_SEED), validation_fraction=np.asarray(.30))
    split_origin = 'deterministic_70_30_source_train'
    del clean_source, noisy_source

# Regenerate train/validation noisy pairs from all three NSTDB artifacts.
# n_type=1: first temporal half, first channel. Test remains the locked NPZ test.
import wfdb
from scipy import signal

MIX_SEED = 1234
MIX_LENGTH = 3600
NOISE_TRAIN = {}
for artifact in ('BW', 'EM', 'MA'):
    record = wfdb.rdrecord(artifact.lower(), pn_dir='nstdb/1.0.0')
    values = np.asarray(record.p_signal, dtype=np.float32)
    if values.ndim != 2 or values.shape[1] < 2:
        raise ValueError(f'NSTDB {artifact}: expected two channels, got {values.shape}')
    if int(record.fs) != 360:
        target_length = int(round(len(values) * 360 / float(record.fs)))
        values = signal.resample(values, target_length, axis=0).astype(np.float32)
    midpoint = len(values) // 2
    NOISE_TRAIN[artifact] = np.ascontiguousarray(values[:midpoint, 0])
    if len(NOISE_TRAIN[artifact]) < MIX_LENGTH or not np.isfinite(NOISE_TRAIN[artifact]).all():
        raise ValueError(f'Invalid NSTDB {artifact} train region')
    print(f'NSTDB {artifact}: ch1_a, {len(NOISE_TRAIN[artifact])} samples')

def make_three_artifact_pairs(clean, q, seed, split_name):
    rng = np.random.RandomState(seed)
    q = np.asarray(q, dtype=np.float32)
    if len(q) != len(clean) or not np.isfinite(q).all() or np.any((q < .20) | (q > 1.99)):
        raise ValueError(f'{split_name}: invalid q levels')
    mixed_noisy = np.empty_like(clean, dtype=np.float32)
    audit = []
    for index in tqdm(range(len(clean)), desc=f'Mix BW+EM+MA {split_name}'):
        components = {}
        starts = {}
        for artifact in ('BW', 'EM', 'MA'):
            source = NOISE_TRAIN[artifact]
            start = int(rng.randint(0, len(source) - MIX_LENGTH + 1))
            starts[artifact] = start
            components[artifact] = source[start:start + MIX_LENGTH]
        a1 = int(rng.randint(0, 101))
        a2 = int(rng.randint(0, 101))
        # Exact TFCDiff expression: only one /100, outside the whole sum.
        mix = (a1 * components['BW'] + (100 - a1) *
               (a2 * components['EM'] + (100 - a2) * components['MA'])) / 100.0
        clean_row = clean[index, 0]
        mix_range = float(np.ptp(mix))
        clean_range = float(np.ptp(clean_row))
        if mix_range <= 0 or clean_range <= 0:
            raise ValueError(f'{split_name} row {index}: flat clean or mixed noise')
        mixed_noisy[index, 0] = clean_row + float(q[index]) * clean_range / mix_range * mix
        audit.append((index, a1, a2, starts['BW'], starts['EM'], starts['MA'], float(q[index])))
    if not np.isfinite(mixed_noisy).all():
        raise FloatingPointError(f'{split_name}: nonfinite noisy pairs')
    audit_frame = pd.DataFrame(audit, columns=['row', 'a1', 'a2', 'bw_start', 'em_start', 'ma_start', 'q'])
    audit_frame.to_csv(REPORT_DIR / f'{split_name}_mix_manifest.csv', index=False)
    return mixed_noisy

del NOISY_TRAIN, NOISY_VAL
NOISY_TRAIN = make_three_artifact_pairs(CLEAN_TRAIN, TRAIN_MANIFEST.q.to_numpy(), MIX_SEED, 'train')
NOISY_VAL = make_three_artifact_pairs(CLEAN_VAL, VAL_MANIFEST.q.to_numpy(), MIX_SEED + 1, 'validation')
print('Generated mixed BW+EM+MA pairs:', NOISY_TRAIN.shape, NOISY_VAL.shape)

if CLEAN_TRAIN.shape != NOISY_TRAIN.shape or CLEAN_VAL.shape != NOISY_VAL.shape:
    raise ValueError('Train/validation clean-noisy shape mismatch')
known_train_records = set(TRAIN_MANIFEST.record_id) - {'unknown'}
known_val_records = set(VAL_MANIFEST.record_id) - {'unknown'}
if known_train_records & TEST_RECORDS or known_val_records & TEST_RECORDS:
    raise RuntimeError('Record leakage: test record xuất hiện trong train/validation')

CLEAN_TEST_LOCKED = as_ncl3600(train_bundle['clean_test'], 'clean_test')
NOISY_TEST_LOCKED = as_ncl3600(train_bundle['noisy_test'], 'noisy_test')
if CLEAN_TEST_LOCKED.shape != NOISY_TEST_LOCKED.shape:
    raise ValueError('Test clean/noisy shape mismatch')
TEST_MANIFEST_LOCKED = make_frame(train_bundle, 'test', len(CLEAN_TEST_LOCKED), 'test')
actual_test_records = set(TEST_MANIFEST_LOCKED.record_id)
if actual_test_records != {'unknown'} and actual_test_records != TEST_RECORDS:
    raise RuntimeError(f'14 test records không khớp; thiếu={TEST_RECORDS-actual_test_records}, thừa={actual_test_records-TEST_RECORDS}')
if actual_test_records == {'unknown'}:
    print('WARNING: không có record_id_test và không suy ra được từ sample_id_test; không thể audit 14 records.')

manifest = pd.concat([TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST_LOCKED], ignore_index=True)
if manifest.sample_id.duplicated().any():
    raise ValueError('sample_id phải duy nhất trên toàn bộ split')
manifest_origin = f'npz:{split_origin}'

def array_sha256(array):
    value = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(value.shape).encode()); digest.update(str(value.dtype).encode())
    digest.update(value.view(np.uint8))
    return digest.hexdigest()

DATA_FINGERPRINTS = {
    'bundle_file': sha256_file(train_bundle_path),
    'clean_train': array_sha256(CLEAN_TRAIN), 'noisy_train': array_sha256(NOISY_TRAIN),
    'clean_val': array_sha256(CLEAN_VAL), 'noisy_val': array_sha256(NOISY_VAL),
    'clean_test': array_sha256(CLEAN_TEST_LOCKED), 'noisy_test': array_sha256(NOISY_TEST_LOCKED),
    'manifest': hashlib.sha256(manifest.to_csv(index=False).encode('utf-8')).hexdigest(),
}
manifest.to_csv(REPORT_DIR / MANIFEST_NAME, index=False)
data_audit = pd.DataFrame([
    {'split': 'train', 'N': len(CLEAN_TRAIN), 'shape': str(CLEAN_TRAIN.shape), 'records': TRAIN_MANIFEST.record_id.nunique()},
    {'split': 'validation', 'N': len(CLEAN_VAL), 'shape': str(CLEAN_VAL.shape), 'records': VAL_MANIFEST.record_id.nunique()},
    {'split': 'test', 'N': len(CLEAN_TEST_LOCKED), 'shape': str(CLEAN_TEST_LOCKED.shape), 'records': TEST_MANIFEST_LOCKED.record_id.nunique()},
])
data_audit.to_csv(REPORT_DIR / 'data_audit.csv', index=False)
display(data_audit)
print('Protocol:', protocol_name, '| verified:', protocol_verified,
      '| n_type=1 | split:', split_origin)
# Audit the frozen 70/30 source indices shipped in this mixed bundle.
if {'train_idx', 'val_idx'}.issubset(train_bundle.files):
    source_train_idx = np.asarray(train_bundle['train_idx'], dtype=np.int64).reshape(-1)
    source_val_idx = np.asarray(train_bundle['val_idx'], dtype=np.int64).reshape(-1)
    if len(source_train_idx) != len(CLEAN_TRAIN) or len(source_val_idx) != len(CLEAN_VAL):
        raise ValueError('train_idx/val_idx length mismatch')
    if len(np.unique(source_train_idx)) != len(source_train_idx) or len(np.unique(source_val_idx)) != len(source_val_idx):
        raise ValueError('Duplicate source indices in mixed split')
    if np.intersect1d(source_train_idx, source_val_idx).size:
        raise ValueError('Source train/validation indices overlap')
    np.savez(REPORT_DIR / 'source_split_indices.npz',
             train_idx=source_train_idx, val_idx=source_val_idx)
noise_version = int(scalar(train_bundle, 'noise_version', -1))
if noise_version != 1:
    raise ValueError(f'Expected noise_version=1, got {noise_version}')
(REPORT_DIR / 'mixed_input_audit.json').write_text(
    json.dumps({'bundle_sha256': DATA_FINGERPRINTS['bundle_file'],
                'protocol_verified': protocol_verified, 'generated_train_mixture': True,
                'mixed_train_seed': MIX_SEED, 'noise_version': noise_version,
                'sampling_rate': sampling_rate, 'n_type': n_type,
                'train_count': len(CLEAN_TRAIN), 'val_count': len(CLEAN_VAL),
                'test_count': len(CLEAN_TEST_LOCKED)}, indent=2), encoding='utf-8')
train_bundle.close()


## 5. Chuẩn hóa chỉ từ noisy input

In [ ]:
NORMALIZATION_EPS = 1e-8

def noisy_scale(noisy):
    scale = np.max(np.abs(noisy), axis=(1, 2), keepdims=True).astype(np.float32)
    if (scale <= NORMALIZATION_EPS).any():
        bad = np.flatnonzero(scale.reshape(-1) <= NORMALIZATION_EPS)[:10]
        raise ValueError(f'Noisy maxabs bằng 0 tại rows {bad.tolist()}')
    return scale

class CommonPairDataset(Dataset):
    def __init__(self, clean, noisy):
        self.clean = clean
        self.noisy = noisy
        self.scale = noisy_scale(noisy)

    def __len__(self):
        return len(self.clean)

    def __getitem__(self, index):
        scale = self.scale[index]
        clean = torch.from_numpy((self.clean[index] / scale).astype(np.float32))
        noisy = torch.from_numpy((self.noisy[index] / scale).astype(np.float32))
        return clean, noisy

train_dataset = CommonPairDataset(CLEAN_TRAIN, NOISY_TRAIN)
val_dataset = CommonPairDataset(CLEAN_VAL, NOISY_VAL)
normalization_audit = pd.DataFrame({
    'split': ['train', 'validation'],
    'scale_min': [train_dataset.scale.min(), val_dataset.scale.min()],
    'scale_median': [np.median(train_dataset.scale), np.median(val_dataset.scale)],
    'scale_max': [train_dataset.scale.max(), val_dataset.scale.max()],
})
normalization_audit.to_csv(REPORT_DIR / 'normalization_audit.csv', index=False)
display(normalization_audit)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_dataset.scale.reshape(-1), bins=50, alpha=.65, label='train', color='#3568A8')
ax.hist(val_dataset.scale.reshape(-1), bins=50, alpha=.55, label='validation', color='#D39C2C')
ax.set(title='Noisy-derived max-abs scale', xlabel='Scale (đơn vị tín hiệu gốc)', ylabel='Số segment')
ax.legend(); ax.grid(axis='y', alpha=.2)
fig.tight_layout(); fig.savefig(REPORT_DIR / 'normalization_scale_distribution.png', dpi=160)
plt.show()


## 6. Preflight: length, STFT và strict state dict

In [ ]:
import torch.nn.functional as F
from src.models.main_model import default

# Extracted verbatim from kaggle_train_qtdb_bw_em_ma.ipynb.
RESEARCH_LOSS_SHA256 = '4b444b6e37bfb5385fca4f49e783f447fcd51c17b67d9345b494f4223c773714'
assert RESEARCH_LOSS_SHA256 == MODEL_CONFIG['experiment']['research_loss_sha256']

class ResearchLossDDPM(DDPM):
    """Same DDPM/UNet architecture, with a scale-stable multi-domain objective."""

    def loss_components(self, x_in, y_in, noise=None):
        x_start = x_in
        batch = x_start.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=x_start.device)
        boundaries = torch.as_tensor(
            self.sqrt_alphas_cumprod_prev, dtype=x_start.dtype, device=x_start.device
        )
        lower, upper = boundaries[timestep], boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=x_start.device) * (upper - lower)
        continuous_2d = continuous.view(batch, 1)
        continuous_3d = continuous.view(batch, 1, 1)
        gaussian_noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(x_start, continuous_3d, gaussian_noise)
        predicted_noise = self.model(x_noisy, y_in, continuous_2d)

        # Mean keeps the time loss independent of batch size and 512-sample length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Reconstruct x0 with the exact continuous alpha used to create x_t.
        residual_scale = torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0))
        x0_prediction = (x_noisy - residual_scale * predicted_noise) / continuous_3d.clamp_min(1e-6)

        n_fft, hop_length = 128, 64
        window = torch.hann_window(n_fft, dtype=x_start.dtype, device=x_start.device)
        pred_flat = x0_prediction.reshape(-1, x0_prediction.shape[-1])
        target_flat = x_start.reshape(-1, x_start.shape[-1])
        pred_mag = torch.abs(torch.stft(pred_flat, n_fft=n_fft, hop_length=hop_length,
                                         window=window, return_complex=True))
        target_mag = torch.abs(torch.stft(target_flat, n_fft=n_fft, hop_length=hop_length,
                                           window=window, return_complex=True))
        pred_mag = pred_mag.reshape(batch, -1)
        target_mag = target_mag.reshape(batch, -1)
        spectral_error = torch.mean((pred_mag - target_mag) ** 2, dim=1)
        target_power = torch.mean(target_mag ** 2, dim=1).clamp_min(1e-6)
        relative_frequency = spectral_error / target_power

        # x0 is ill-conditioned at high-noise timesteps, so weight STFT by diffusion SNR.
        alpha_bar = continuous ** 2
        diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
        frequency_weight = torch.clamp(diffusion_snr, max=1.0)
        loss_frequency = torch.mean(relative_frequency * frequency_weight)
        total = self.lambda_time * loss_time + self.lambda_freq * loss_frequency
        return {'total': total, 'time': loss_time, 'frequency': loss_frequency,
                'frequency_unweighted': relative_frequency.mean(),
                'frequency_weight': frequency_weight.mean()}

    def p_losses(self, x_in, y_in, noise=None):
        return self.loss_components(x_in, y_in, noise)['total']

def build_model():
    base = UNet1D(in_channels=2, base_channels=MODEL_CONFIG['train']['feats'],
                  emb_dim=128, out_channels=1)
    return ResearchLossDDPM(base, MODEL_CONFIG, DEVICE).to(DEVICE)

model = build_model()
parameter_count = sum(parameter.numel() for parameter in model.parameters())
with torch.no_grad():
    probe = torch.sin(torch.linspace(0, 8 * torch.pi, 3600, device=DEVICE)).view(1, 1, 3600)
    alpha = torch.full((1, 1), 0.8, device=DEVICE)
    direct_output = model.model(probe, probe, alpha)
    if direct_output.shape != probe.shape:
        raise RuntimeError(f'U-Net shape sai: {direct_output.shape}')
    research_probe = model.loss_components(probe, probe)
    if not all(torch.isfinite(value) for value in research_probe.values()):
        raise FloatingPointError('Research loss preflight non-finite')

strict_probe = build_model()
strict_probe.load_state_dict(model.state_dict(), strict=True)
del strict_probe, probe, direct_output, research_probe
torch.cuda.empty_cache()
print(f'Preflight OK: [1,1,3600] -> [1,1,3600]; params={parameter_count:,}; '
      '3 downsamplings: 3600 -> 1800 -> 900 -> 450; research STFT(128,64) OK; strict=True OK')


## 7. DataLoaders, EMA và checkpoint/resume

In [ ]:
LAST_NAME = 'last_qtdb_mixed_bwem_ma_3600_fresh.pth'
BEST_NAME = 'best_qtdb_mixed_bwem_ma_3600_fresh.pth'
LOG_NAME = 'training_qtdb_mixed_bwem_ma_3600_fresh.csv'

def make_loader(dataset, shuffle, epoch=0):
    generator = torch.Generator().manual_seed(SEED + epoch)
    return DataLoader(dataset, batch_size=MICRO_BATCH_SIZE, shuffle=shuffle,
                      drop_last=True, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=NUM_WORKERS > 0,
                      generator=generator)

def cpu_state_dict(state):
    return {key: value.detach().cpu().clone() for key, value in state.items()}

def ema_model_state(model, ema):
    state = cpu_state_dict(model.state_dict())
    for key, value in ema.state_dict().items():
        state[key] = value.detach().cpu().clone()
    return state

def optimizer_to(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

def checkpoint_candidates(filename):
    local = CHECKPOINT_DIR / filename
    candidates = ([local] if local.is_file() else []) + sorted(INPUT_ROOT.rglob(filename))
    # The local copy and its source input may coexist only after an in-notebook save.
    unique = []
    for path in candidates:
        if path not in unique:
            unique.append(path)
    return unique

def resolve_resume_checkpoint():
    candidates = checkpoint_candidates(LAST_NAME)
    if not candidates:
        return None
    if len(candidates) > 1 and (CHECKPOINT_DIR / LAST_NAME) not in candidates:
        raise RuntimeError(f'Nhiều resume checkpoint: {candidates}')
    return CHECKPOINT_DIR / LAST_NAME if (CHECKPOINT_DIR / LAST_NAME).is_file() else candidates[0]

def resolve_named_checkpoint(filename):
    local = CHECKPOINT_DIR / filename
    if local.is_file():
        return local
    matches = sorted(INPUT_ROOT.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Cần đúng một {filename}: {matches}')
    return matches[0]

def atomic_torch_save(payload, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

optimizer = torch.optim.Adam(model.parameters(), lr=MODEL_CONFIG['train']['lr'], weight_decay=MODEL_CONFIG['train']['weight_decay'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=150, gamma=0.1)
ema = EMA(EMA_DECAY)
ema.register(model)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
start_epoch, best_validation_loss = 0, float('inf')
resume_path = None if START_FROM_SCRATCH else resolve_resume_checkpoint()
if REQUIRE_RESUME_CHECKPOINT and resume_path is None:
    raise FileNotFoundError(
        f'Không tìm thấy {LAST_NAME} trong Kaggle Input. Gắn checkpoint đã giải nén '
        'trước khi Run All; notebook sẽ không khởi tạo run mới.')

if resume_path is not None:
    payload = torch.load(resume_path, map_location='cpu', weights_only=False)
    if payload['config'] != MODEL_CONFIG:
        raise RuntimeError('Resume config khác phiên hiện tại')
    if payload['data_fingerprints'] != DATA_FINGERPRINTS:
        raise RuntimeError('Resume data fingerprint mismatch')
    if payload['source_hashes'] != source_hashes:
        raise RuntimeError('Resume source hash mismatch')
    model.load_state_dict(payload['state_dict'], strict=True)
    optimizer.load_state_dict(payload['optimizer_state_dict']); optimizer_to(optimizer, DEVICE)
    scheduler.load_state_dict(payload['scheduler_state_dict'])
    scaler.load_state_dict(payload['scaler_state_dict'])
    ema.load_state_dict({key: value.to(DEVICE) for key, value in payload['ema_state_dict'].items()})
    torch.set_rng_state(payload['torch_rng_state'])
    torch.cuda.set_rng_state(payload['cuda_rng_state'])
    start_epoch = int(payload['epoch']) + 1
    best_validation_loss = float(payload['best_validation_loss'])
    previous_log = resume_path.parent / LOG_NAME
    if not previous_log.is_file():
        log_matches = sorted(INPUT_ROOT.rglob(LOG_NAME))
        if len(log_matches) > 1:
            raise RuntimeError(f'Nhiều training log: {log_matches}')
        previous_log = log_matches[0] if log_matches else previous_log
    if previous_log.is_file():
        shutil.copy2(previous_log, REPORT_DIR / LOG_NAME)
    previous_best = resume_path.parent / BEST_NAME
    if not previous_best.is_file():
        best_matches = sorted(INPUT_ROOT.rglob(BEST_NAME))
        if len(best_matches) > 1:
            raise RuntimeError(f'Nhiều best checkpoint: {best_matches}')
        previous_best = best_matches[0] if best_matches else previous_best
    if previous_best.is_file():
        shutil.copy2(previous_best, CHECKPOINT_DIR / BEST_NAME)
    else:
        print('WARNING: Chỉ tìm thấy last checkpoint; thiếu best checkpoint trước đây. '
              'Training vẫn tiếp tục, nhưng cần best gốc để giữ epoch tốt nhất trước resume.')
    print(f'Resume từ {resume_path}: epoch {start_epoch}/{TARGET_EPOCHS}')
else:
    print('Khởi tạo training run mới')

def save_checkpoint(epoch, is_best=False):
    payload = {
        'state_dict': cpu_state_dict(model.state_dict()),
        'ema_state_dict': cpu_state_dict(ema.state_dict()),
        'ema_model_state_dict': ema_model_state(model, ema),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'epoch': epoch, 'best_validation_loss': best_validation_loss,
        'config': MODEL_CONFIG, 'data_fingerprints': DATA_FINGERPRINTS,
        'source_hashes': source_hashes, 'parameter_count': parameter_count,
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state(),
    }
    atomic_torch_save(payload, CHECKPOINT_DIR / (BEST_NAME if is_best else LAST_NAME))


## 8. Huấn luyện theo phiên và chọn best chỉ bằng validation

In [ ]:
# Stop if repeated non-finite gradients indicate a persistent problem.
MAX_NONFINITE_STEPS_PER_EPOCH = 5

def validation_loss_fixed(model, epoch):
    cpu_rng = torch.get_rng_state()
    cuda_rng = torch.cuda.get_rng_state()
    torch.manual_seed(SEED + 1_000_000)
    torch.cuda.manual_seed_all(SEED + 1_000_000)
    model.eval()
    sums = {'total': 0.0, 'time': 0.0, 'frequency': 0.0,
            'frequency_unweighted': 0.0, 'frequency_weight': 0.0}
    examples = 0
    loader = make_loader(val_dataset, shuffle=False, epoch=0)
    try:
        with torch.no_grad():
            for clean, noisy in tqdm(loader, desc=f'Validation epoch {epoch + 1}', leave=False):
                clean = clean.to(DEVICE, non_blocking=True)
                noisy = noisy.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    components = model.loss_components(clean, noisy)
                if not all(torch.isfinite(value) for value in components.values()):
                    raise FloatingPointError('Validation loss NaN/Inf')
                for name in sums:
                    sums[name] += float(components[name]) * len(clean)
                examples += len(clean)
    finally:
        torch.set_rng_state(cpu_rng)
        torch.cuda.set_rng_state(cuda_rng)
    if examples == 0:
        raise RuntimeError('Validation set nhỏ hơn micro-batch')
    return {name: value / examples for name, value in sums.items()}

def append_log(row):
    path = REPORT_DIR / LOG_NAME
    pd.DataFrame([row]).to_csv(path, mode='a', header=not path.exists(), index=False)

session_end = TARGET_EPOCHS if MAX_EPOCHS_THIS_SESSION is None else min(
    TARGET_EPOCHS, start_epoch + MAX_EPOCHS_THIS_SESSION)

if MODE == 'train':
    if start_epoch >= TARGET_EPOCHS:
        print('Training đã đủ target; không chạy thêm epoch.')
    for epoch in range(start_epoch, session_end):
        model.train()
        sums = {'total': 0.0, 'time': 0.0, 'frequency': 0.0}
        examples = 0
        gradient_norms = []
        skipped_steps = 0
        nonfinite_gradient_steps = 0
        loader = make_loader(train_dataset, shuffle=True, epoch=epoch)
        usable_batches = (len(loader) // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        if usable_batches == 0:
            raise RuntimeError('Train set nhỏ hơn effective batch 96')
        progress = tqdm(loader, desc=f'Epoch {epoch + 1}/{TARGET_EPOCHS}')
        optimizer.zero_grad(set_to_none=True)
        for batch_index, (clean, noisy) in enumerate(progress):
            if batch_index >= usable_batches:
                break
            clean = clean.to(DEVICE, non_blocking=True)
            noisy = noisy.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                components = model.loss_components(clean, noisy)
                backward_loss = components['total'] / GRAD_ACCUM_STEPS
            if not all(torch.isfinite(value) for value in components.values()):
                raise FloatingPointError(f'Loss NaN/Inf tại epoch {epoch + 1}')
            scaler.scale(backward_loss).backward()
            if (batch_index + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(gradient_norm):
                    # Do not step the optimizer or EMA with invalid gradients.
                    # clip_grad_norm_ may have touched them; zero them immediately.
                    nonfinite_gradient_steps += 1
                    skipped_steps += 1
                    old_scale = scaler.get_scale()
                    if USE_AMP:
                        scaler.update(new_scale=max(old_scale * 0.5, 1.0))
                    else:
                        scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    print(f'WARNING epoch {epoch + 1}, batch {batch_index + 1}: '
                          f'gradient norm={float(gradient_norm)}; skipped update; '
                          f'AMP scale {old_scale:g} -> {scaler.get_scale():g}')
                    if nonfinite_gradient_steps > MAX_NONFINITE_STEPS_PER_EPOCH:
                        raise FloatingPointError(
                            f'Quá {MAX_NONFINITE_STEPS_PER_EPOCH} bước gradient NaN/Inf '
                            f'tại epoch {epoch + 1}; cần kiểm tra loss/AMP thay vì tiếp tục im lặng.')
                else:
                    gradient_norms.append(float(gradient_norm))
                    old_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale() >= old_scale:
                        ema.update(model)
                    else:
                        skipped_steps += 1
                    optimizer.zero_grad(set_to_none=True)
            for name in sums:
                sums[name] += float(components[name].detach()) * len(clean)
            examples += len(clean)
            progress.set_postfix(loss=f'{sums["total"] / examples:.5f}')
        scheduler.step()
        train_values = {name: value / examples for name, value in sums.items()}
        val_values = validation_loss_fixed(model, epoch)
        val_loss = val_values['total']
        improved = val_loss < best_validation_loss
        if improved:
            best_validation_loss = val_loss
        append_log({
            'epoch': epoch + 1, 'train_loss': train_values['total'],
            'train_time_loss': train_values['time'],
            'train_frequency_loss': train_values['frequency'],
            'validation_loss': val_loss,
            'validation_time_loss': val_values['time'],
            'validation_frequency_loss': val_values['frequency'],
            'validation_frequency_unweighted': val_values['frequency_unweighted'],
            'validation_frequency_weight': val_values['frequency_weight'],
            'best_validation_loss': best_validation_loss,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'gradient_norm_mean': float(np.mean(gradient_norms)),
            'gradient_norm_max': float(np.max(gradient_norms)),
            'nonfinite_gradient_steps': nonfinite_gradient_steps,
            'amp_skipped_steps': skipped_steps,
            'micro_batch_size': MICRO_BATCH_SIZE,
            'effective_batch_size': REFERENCE_BATCH_SIZE,
            'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
            'amp': USE_AMP,
        })
        save_checkpoint(epoch, is_best=False)
        if improved:
            save_checkpoint(epoch, is_best=True)
        print(f'Epoch {epoch + 1}: train={train_values["total"]:.6f}; '
              f'val={val_loss:.6f}; best={best_validation_loss:.6f}'
              + (' [BEST]' if improved else ''))
        gc.collect(); torch.cuda.empty_cache()

last_candidates = checkpoint_candidates(LAST_NAME)
completed_epochs = start_epoch if not last_candidates else int(
    torch.load(last_candidates[0], map_location='cpu', weights_only=False)['epoch']) + 1
TRAINING_COMPLETE = completed_epochs >= TARGET_EPOCHS
print(f'Đã hoàn thành {completed_epochs}/{TARGET_EPOCHS} epoch. '
      + ('Có thể export.' if TRAINING_COMPLETE else
         'Save Version, gắn output làm Input và chạy lại để resume.'))


## 9. Đường học của loss nghiên cứu

Log ghi riêng time loss, frequency loss, gradient norm và số bước AMP bỏ qua; dùng các cột này để kiểm tra ổn định qua các phiên.


In [ ]:
log_path = REPORT_DIR / LOG_NAME
if log_path.is_file():
    training_log = pd.read_csv(log_path).drop_duplicates('epoch', keep='last').sort_values('epoch')
    training_log.to_csv(log_path, index=False)
    display(training_log.tail(10))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].plot(training_log.epoch, training_log.train_loss, label='Train', color='#3568A8')
    axes[0].plot(training_log.epoch, training_log.validation_loss, label='Validation', color='#D39C2C')
    axes[0].plot(training_log.epoch, training_log.best_validation_loss,
                 label='Best validation', color='#333333', linestyle='--')
    axes[0].set(title='Total research loss', xlabel='Epoch', ylabel='Normalized loss')
    axes[0].legend()
    axes[1].plot(training_log.epoch, training_log.train_time_loss,
                 label='Time (train)', color='#3568A8')
    axes[1].plot(training_log.epoch, training_log.validation_time_loss,
                 label='Time (validation)', color='#D39C2C')
    axes[1].plot(training_log.epoch, training_log.train_frequency_loss,
                 label='Frequency (train)', color='#3568A8', linestyle='--')
    axes[1].plot(training_log.epoch, training_log.validation_frequency_loss,
                 label='Frequency (validation)', color='#D39C2C', linestyle='--')
    axes[1].set(title='Loss components before lambda', xlabel='Epoch', ylabel='Component loss')
    axes[1].legend(fontsize=8)
    axes[2].plot(training_log.epoch, training_log.gradient_norm_mean,
                 label='Mean', color='#3568A8')
    axes[2].plot(training_log.epoch, training_log.gradient_norm_max,
                 label='Maximum', color='#D39C2C', alpha=.7)
    axes[2].set(title='Gradient norm before clipping', xlabel='Epoch', ylabel='L2 norm')
    axes[2].legend()
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle('QTDB mixed BW+EM+MA — Research DDPM', fontsize=13)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'training_history.png', dpi=180, bbox_inches='tight')
    plt.show()


## 10. Mở test chỉ sau khi checkpoint đã được khóa

Test tensors đã được fingerprint ở bước data audit, nhưng không đi vào DataLoader train/validation và không tham gia chọn checkpoint.


In [ ]:
def load_test_contract():
    return (NOISY_TEST_LOCKED,
            TEST_MANIFEST_LOCKED.sample_id.astype(str).to_numpy(),
            TEST_MANIFEST_LOCKED.copy())

SHOULD_EXPORT = MODE == 'export' or (MODE == 'train' and RUN_EXPORT_WHEN_COMPLETE and TRAINING_COMPLETE)
if SHOULD_EXPORT and not TRAINING_COMPLETE and not ALLOW_INCOMPLETE_EXPORT:
    raise RuntimeError(f'Checkpoint mới có {completed_epochs}/{TARGET_EPOCHS} epoch; không export test.')
print('Export requested:', SHOULD_EXPORT)


## 11. DDIM 1-shot và export `qtdb_mixed_bwem_ma_3600_fresh_seed1234.npz`

In [ ]:
if SHOULD_EXPORT:
    best_path = resolve_named_checkpoint(BEST_NAME)
    best_payload = torch.load(best_path, map_location='cpu', weights_only=False)
    if best_payload['config'] != MODEL_CONFIG:
        raise RuntimeError('Best checkpoint config mismatch')
    if best_payload['data_fingerprints'] != DATA_FINGERPRINTS:
        raise RuntimeError('Best checkpoint data mismatch')
    if best_payload['source_hashes'] != source_hashes:
        raise RuntimeError('Best checkpoint source mismatch')

    inference_model = build_model()
    inference_model.load_state_dict(best_payload['ema_model_state_dict'], strict=True)
    inference_model.eval()
    NOISY_TEST, SAMPLE_ID_TEST, TEST_MANIFEST = load_test_contract()
    TEST_MANIFEST.to_csv(REPORT_DIR / 'test_manifest_locked.csv', index=False)
    test_scale = noisy_scale(NOISY_TEST)
    predictions, latency = [], []
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    with torch.inference_mode():
        for start in tqdm(range(0, len(NOISY_TEST), INFERENCE_MICRO_BATCH_SIZE), desc='DDIM 1-shot'):
            stop = min(start + INFERENCE_MICRO_BATCH_SIZE, len(NOISY_TEST))
            condition = torch.from_numpy(
                (NOISY_TEST[start:stop] / test_scale[start:stop]).astype(np.float32)
            ).to(DEVICE)
            torch.cuda.synchronize(); began = time.perf_counter()
            prediction_norm = inference_model.denoising(
                condition, use_ddim=True, ddim_steps=DDIM_STEPS,
                ddim_eta=DDIM_ETA, num_shots=NUM_SHOTS)
            torch.cuda.synchronize(); elapsed_ms = (time.perf_counter() - began) * 1000
            prediction = prediction_norm.cpu().numpy() * test_scale[start:stop]
            predictions.append(prediction.astype(np.float32))
            latency.extend([elapsed_ms / (stop - start)] * (stop - start))
    prediction = np.concatenate(predictions, axis=0)
    latency_ms = np.asarray(latency, dtype=np.float32)
    if prediction.shape != NOISY_TEST.shape or prediction.shape[1:] != (1, 3600):
        raise RuntimeError(f'Prediction shape sai: {prediction.shape}')
    if not np.isfinite(prediction).all() or not np.isfinite(latency_ms).all():
        raise FloatingPointError('Prediction/latency có NaN/Inf')
    if len(np.unique(SAMPLE_ID_TEST)) != len(SAMPLE_ID_TEST):
        raise ValueError('Test sample_id không duy nhất')

    prediction_path = WORK_ROOT / 'qtdb_mixed_bwem_ma_3600_fresh_seed1234.npz'
    np.savez_compressed(prediction_path, sample_id=SAMPLE_ID_TEST,
                        prediction=prediction, latency_ms=latency_ms,
                        normalization_scale=test_scale.reshape(-1))
    export_audit = {
        'prediction_file': str(prediction_path), 'N': len(prediction),
        'prediction_shape': list(prediction.shape), 'sample_id_order_preserved': True,
        'finite': True, 'checkpoint': str(best_path),
        'checkpoint_sha256': sha256_file(best_path),
        'selected_epoch': int(best_payload['epoch']) + 1,
        'validation_loss': float(best_payload['best_validation_loss']),
        'sampler': 'DDIM', 'ddim_steps': DDIM_STEPS, 'ddim_eta': DDIM_ETA,
        'num_shots': NUM_SHOTS, 'seed': SEED,
        'latency_ms_mean': float(latency_ms.mean()),
    }
    (REPORT_DIR / 'export_audit.json').write_text(
        json.dumps(export_audit, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps(export_audit, indent=2, ensure_ascii=False))


## 12. Metadata và gói output

In [ ]:
run_info = {
    'experiment': 'qtdb_mixed_bwem_ma_3600_fresh_generated_train',
    'mode': MODE, 'target_epochs': TARGET_EPOCHS,
    'completed_epochs': completed_epochs, 'training_complete': TRAINING_COMPLETE,
    'config': MODEL_CONFIG, 'device': str(DEVICE),
    'gpu': torch.cuda.get_device_name(0), 'torch_version': torch.__version__,
    'python_version': platform.python_version(), 'parameter_count': parameter_count,
    'data_fingerprints': DATA_FINGERPRINTS, 'source_hashes': source_hashes,
    'common_bundle': str(train_bundle_path), 'manifest_origin': manifest_origin, 'protocol_verified': protocol_verified, 'generated_train_mixture': True, 'mixed_train_seed': MIX_SEED, 'noise_version': noise_version, 'n_type': n_type, 'sampling_rate': sampling_rate, 'segment_length': segment_length,
    'normalization_scale_source': 'noisy input only',
    'test_used_for_checkpoint_selection': False,
    'metrics_computed_in_model_notebook': False,
}
(REPORT_DIR / 'run_info.json').write_text(
    json.dumps(run_info, indent=2, ensure_ascii=False), encoding='utf-8')

archive = Path('/kaggle/working/qtdb_mixed_bwem_ma_3600_fresh_outputs.zip')
if archive.exists():
    archive.unlink()
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output_zip:
    for path in sorted(WORK_ROOT.rglob('*')):
        if path.is_file():
            output_zip.write(path, path.relative_to(WORK_ROOT))
print('Output directory:', WORK_ROOT)
print('Archive:', archive, f'({archive.stat().st_size / 1024**2:.1f} MiB)')


## Chạy từ đầu và lưu kết quả

Trong phiên Kaggle đầu, để `START_FROM_SCRATCH=True`: notebook khởi tạo model mới ở epoch 0, không cần gắn `last_*.pth`. `MAX_EPOCHS_THIS_SESSION=5` là giới hạn an toàn để đo thời gian; tăng sau khi xác nhận thời gian/epoch. Notebook vẫn lưu `last` và `best` sau mỗi epoch để có đường phục hồi nếu Kaggle hết thời gian. Nếu cần tiếp tục qua phiên khác, gắn output, đặt `START_FROM_SCRATCH=False`; khi đó checkpoint, optimizer, EMA, AMP scaler và RNG được khôi phục.

Train/validation: QTDB clean 3600 từ NPZ + hỗn hợp BW/EM/MA được tạo trong notebook. Test: **chỉ** `noisy_test` cố định trong NPZ; prediction được inverse-normalize và xuất theo `sample_id_test`. Không chọn checkpoint bằng test.
